# Set Covering Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.15


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.


## Problem Statement

Book problem `2.15` has two binary set-covering models built on the same city map:

1. part `a`: choose fire-station locations to cover all `22` neighborhoods at minimum fixed cost,
2. part `b`: choose at most `4` locations to maximize the population covered according to the printed
   objective in the book.

### Note on part (b)

The printed objective counts a neighborhood once for every selected station that can cover it. That is
the exact model reproduced here. At the end of the notebook we also report, as a classroom note, the
best `unique-population` coverage obtained by four stations.


In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

try:
    from amplpy import AMPL
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install amplpy in the active environment before running this notebook.") from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


NEIGHBORHOODS = list(range(1, 23))
STATIONS = list(range(1, 25))
COVERAGE = {1: [1, 2],
 2: [1, 19],
 3: [2, 3, 19],
 4: [19, 20],
 5: [10, 17, 19],
 6: [18, 19],
 7: [17, 18, 22],
 8: [17, 22],
 9: [15, 16, 17],
 10: [10, 11, 16],
 11: [13, 15, 16],
 12: [14, 15, 21],
 13: [14, 21],
 14: [12, 13],
 15: [7, 13, 14, 21],
 16: [7, 21],
 17: [8, 9, 10, 11],
 18: [3, 4, 8, 9],
 19: [2, 3, 4],
 20: [4, 5],
 21: [7, 8],
 22: [5, 6, 7],
 23: [5, 6],
 24: [2, 4, 6]}
FIXED_COST = {1: 529,
 2: 678,
 3: 816,
 4: 866,
 5: 370,
 6: 357,
 7: 788,
 8: 767,
 9: 640,
 10: 340,
 11: 575,
 12: 387,
 13: 565,
 14: 503,
 15: 731,
 16: 759,
 17: 827,
 18: 555,
 19: 412,
 20: 600,
 21: 810,
 22: 568,
 23: 776,
 24: 798}
POPULATION = {1: 65333,
 2: 103022,
 3: 44088,
 4: 56933,
 5: 100358,
 6: 166906,
 7: 78097,
 8: 96991,
 9: 47830,
 10: 103087,
 11: 98172,
 12: 67638,
 13: 169659,
 14: 151544,
 15: 126496,
 16: 94383,
 17: 103975,
 18: 100276,
 19: 98257,
 20: 65440,
 21: 92170,
 22: 8971}
A_PARAM = {(station, neighborhood): int(neighborhood in COVERAGE[station]) for station in STATIONS for neighborhood in NEIGHBORHOODS}


def create_ampl_instance(solver: str = "highs"):
    ampl = AMPL()
    ampl.eval(f"option solver {solver};")
    return ampl


@timed
def solve_cover_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set L ordered;
        set B ordered;
        param C {L} >= 0;
        param A {L, B} binary;
        var x {L} binary;

        minimize TotalCost:
            sum {i in L} C[i] * x[i];

        subject to CoverEveryNeighborhood {j in B}:
            sum {i in L} A[i, j] * x[i] >= 1;
        '''
    )
    ampl.set["L"] = STATIONS
    ampl.set["B"] = NEIGHBORHOODS
    ampl.param["C"] = FIXED_COST
    ampl.param["A"] = A_PARAM
    ampl.solve(solver=solver)
    values = ampl.get_variable("x").get_values().to_dict()
    selected = [station for station in STATIONS if round(values[station]) == 1]
    return {"stations": selected, "total_cost": int(round(ampl.get_value("TotalCost")))}


@timed
def solve_population_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set L ordered;
        set B ordered;
        param P {B} >= 0;
        param A {L, B} binary;
        var x {L} binary;

        maximize PrintedObjective:
            sum {i in L, j in B} P[j] * A[i, j] * x[i];

        subject to MaxFourStations:
            sum {i in L} x[i] <= 4;
        '''
    )
    ampl.set["L"] = STATIONS
    ampl.set["B"] = NEIGHBORHOODS
    ampl.param["P"] = POPULATION
    ampl.param["A"] = A_PARAM
    ampl.solve(solver=solver)
    values = ampl.get_variable("x").get_values().to_dict()
    selected = [station for station in STATIONS if round(values[station]) == 1]
    covered = set()
    for station in selected:
        covered.update(COVERAGE[station])
    return {
        "stations": selected,
        "objective": int(round(ampl.get_value("PrintedObjective"))),
        "unique_population": sum(POPULATION[neighborhood] for neighborhood in covered),
    }


## Part (a): Minimum Cost


In [2]:
cover_result, cover_time = solve_cover_with_ampl()
print("AMPL part (a) result:", cover_result)
print(f"Elapsed time: {cover_time:.6f} seconds")
assert cover_result == {"stations": [1, 4, 7, 10, 12, 14, 18, 22], "total_cost": 4536}


HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 4536
7 simplex iterations
1 branching nodes
AMPL part (a) result: {'stations': [1, 4, 7, 10, 12, 14, 18, 22], 'total_cost': 4536}
Elapsed time: 0.064936 seconds


## Part (b): Printed Objective


In [3]:
population_result, population_time = solve_population_with_ampl()
print("AMPL part (b) result:", population_result)
print(f"Elapsed time: {population_time:.6f} seconds")
assert population_result["stations"] == [11, 12, 15, 17]
assert population_result["objective"] == 1598298
print("\nClassroom note: the printed objective double-counts neighborhoods covered by more than one selected station.")


HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 1598298
1 simplex iterations
1 branching nodes
AMPL part (b) result: {'stations': [11, 12, 15, 17], 'objective': 1598298, 'unique_population': 1058429}
Elapsed time: 0.011870 seconds

Classroom note: the printed objective double-counts neighborhoods covered by more than one selected station.
